In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import numpy as np
import math
import glob
from tqdm import tqdm

# Hyperparameters

In [ ]:
num_epochs = 1
learning_rate=2e-4
batchsize = 1

# Model

In [ ]:
def encoderBlock(input, num_filters, time_emb, hasAttention):
    x = tf.keras.layers.Conv2D(num_filters, 3, padding="same", activation="swish")(input)
    
    # in DDPM paper the t_emb is injected between the two conv
    t = layers.Dense(num_filters)(time_emb)   
    t = t[:, None, None, :]
    x = x + t 
    
    x = tf.keras.layers.Conv2D(num_filters, 3, padding="same", activation="swish")(x)

    if (hasAttention):
        x = AttentionBlock(num_filters)(x)
    skip = x
    
    x = tf.keras.layers.MaxPool2D(pool_size=(2, 2), strides=2)(x)
    
    return x, skip

def decoderBlock(input, num_filters, time_emb, skip_features, hasAttention):
    x = tf.keras.layers.UpSampling2D(size=(2,2))(input)
    x = tf.keras.layers.Concatenate()([x, skip_features])

    x = tf.keras.layers.Conv2D(num_filters, 3, padding="same", activation="swish")(x)

    # in DDPM paper the t_emb is injected between the two conv
    t = layers.Dense(num_filters)(time_emb)
    t = t[:, None, None, :]
    x = x + t
    
    x = tf.keras.layers.Conv2D(num_filters, 3, padding="same", activation="swish")(x)

    if (hasAttention):
        x = AttentionBlock(num_filters)(x)

    return x


def kernel_init(scale):
    scale = max(scale, 1e-10)
    return tf.keras.initializers.VarianceScaling(
        scale, mode="fan_avg", distribution="uniform"
    )

class AttentionBlock(layers.Layer):
    def __init__(self, units, groups=8, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.groups = groups
        self.norm = layers.GroupNormalization(groups=groups)
        self.query = layers.Dense(units, kernel_initializer=kernel_init(1.0))
        self.key = layers.Dense(units, kernel_initializer=kernel_init(1.0))
        self.value = layers.Dense(units, kernel_initializer=kernel_init(1.0))
        self.proj = layers.Dense(units, kernel_initializer=kernel_init(0.0))

    def call(self, inputs):
        batch_size = tf.shape(inputs)[0]
        height = tf.shape(inputs)[1]
        width = tf.shape(inputs)[2]
        scale = tf.cast(self.units, tf.float32) ** (-0.5)

        h = inputs
        inputs = self.norm(inputs)
        q = self.query(inputs)
        k = self.key(inputs)
        v = self.value(inputs)

        attn_score = tf.einsum("bhwc, bHWc->bhwHW", q, k) * scale
        attn_score = tf.reshape(attn_score, [batch_size, height, width, height * width])

        attn_score = tf.nn.softmax(attn_score, -1)
        attn_score = tf.reshape(attn_score, [batch_size, height, width, height, width])

        proj = tf.einsum("bhwHW,bHWc->bhwc", attn_score, v)
        proj = self.proj(proj)
        return h + proj

class TimeEmbedding(layers.Layer):
    def __init__(self, dim, **kwargs):
        super().__init__(**kwargs)
        self.dim = dim
        self.half_dim = dim // 2
        self.emb = math.log(10000) / (self.half_dim - 1)
        self.emb = tf.exp(tf.range(self.half_dim, dtype=tf.float32) * -self.emb)

    def call(self, inputs):
        inputs = tf.cast(inputs, dtype=tf.float32)
        emb = inputs[:, None] * self.emb[None, :]
        emb = tf.concat([tf.sin(emb), tf.cos(emb)], axis=-1)
        return emb
    
def U_Net():
    img_input = tf.keras.layers.Input(shape=(64,64,4),name="image")
    t_input = tf.keras.layers.Input(shape=(), dtype=tf.int32, name="time_input")

    #time embedding
    time_emb = TimeEmbedding(128)(t_input) # 128 t_emb dimension
    time_emb = layers.Dense(128, activation="swish")(time_emb)

    #encoder
    x,s1 = encoderBlock(img_input, 32, time_emb, False)
    x,s2 = encoderBlock(x, 64, time_emb, True)
    x,s3 = encoderBlock(x, 128, time_emb, True)

    #bottleneck
    x = tf.keras.layers.Conv2D(256, 3, padding="same", activation="swish")(x)
    t = layers.Dense(256)(time_emb)   
    t = t[:, None, None, :] # change later ?
    x = x + t 
    x = tf.keras.layers.Conv2D(256, 3, padding="same", activation="swish")(x)
    x = AttentionBlock(256)(x)

    #decoder
    x = decoderBlock(x, 128, time_emb, s3, True)
    x = decoderBlock(x, 64, time_emb, s2, True)
    x = decoderBlock(x, 32, time_emb, s1, False)

    output = layers.Conv2D(4, 1, padding="same")(x)

    model = tf.keras.Model(
        inputs=[img_input,t_input],
        outputs = output,
        name="U-NET",
        
    )
    model.compile(
        loss=tf.keras.losses.MeanSquaredError(),
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate)
    )
    return model

    
# if __name__ == "__main__":
#     model = U_Net()
#     # model.summary()

#     
#     dummy_img = np.random.randn(1, 64, 64, 4).astype("float32")
#     dummy_t   = np.array([100], dtype="float32")      # two different timesteps
#     pred      = model([dummy_img, dummy_t])
#     print("Output shape:", pred.shape)   
#     pred = tf.reshape(pred, (64,64,4))
#     pred = np.clip(pred, a_max=5, a_min=-5)
#     pred = pred  / 10 + 0.5
#     plt.imshow(pred)
#     plt.show

# Dataset

In [ ]:
def decode_fn(file):
    schema = {
        "image":tf.io.FixedLenFeature([], dtype=tf.string),
        "caption":tf.io.FixedLenFeature([], dtype=tf.string)
    }
    
    parsed = tf.io.parse_single_example(file, schema)
    image = tf.io.decode_image(parsed["image"], channels=4, dtype=tf.uint8)
    image= tf.cast(image, tf.float32) / 127.5 - 1
    image = tf.reshape(image, [64,64,4])
    caption = parsed["caption"]

    return (image, caption)

files = glob.glob("../../Datasets/minecraft_tfrecords/*.tfrecord")

dataset = tf.data.TFRecordDataset(files)\
    .map(decode_fn, num_parallel_calls=tf.data.AUTOTUNE)\
    .batch(batchsize, drop_remainder=True)\
    .prefetch(tf.data.AUTOTUNE)\
    .shuffle(1000)

# for img, _ in dataset.take(1):
#     for i in range(3):
#         # print(f"Caption: {cap.numpy()[i].decode('utf-8')}")
#         plt.imshow(tf.reshape(img[i], (64,64,4)) / 2 + 0.5)
#         plt.axis('off')
#         plt.show()

#     img = img[:3,:,:,:]
#     time = tf.constant([100,200,0], dtype="float32")
#     print(f'shape : {img.shape, time.shape}')
#     model = U_Net()
#     pred = model([img, time])
#     print("Output shape:", pred.shape)
#     res = np.zeros((3,64,64,4))
#     for i in range(3):
#         res[i] = np.clip(pred[i], a_max=5, a_min=-5)
#         res[i] = res[i]  / 10 + 0.5
#         plt.imshow(res[i])
#         plt.show()

# Training Utils

In [ ]:
class DiffusionUtils:
    
    def __init__(self):
        self.beta_start=1e-4
        self.beta_end=0.02
        self.timesteps=1000
        self.clip_min=-1.0
        self.clip_max=1.0

        # great video for the math : https://www.youtube.com/watch?v=HoKDTa5jHvg
        betas = np.linspace(
            self.beta_start,
            self.beta_end,
            self.timesteps,
            dtype=np.float32,
        )
        self.betas = tf.constant(betas, dtype=tf.float32)
        self.sqrt_one_minus_betas = np.sqrt(1 - self.betas)
        self.alphas = 1 - self.betas
        self.alphas_cumprod = tf.math.cumprod(self.alphas)
        self.sqrt_alphas_c = tf.constant(tf.sqrt(self.alphas_cumprod))
        self.sqrt_one_minus_alphas_c = tf.constant(tf.sqrt(1 - self.alphas_cumprod))

        self.one_over_sqrt_alphas = tf.constant(1 / tf.sqrt(self.alphas))

        

    def apply_noise(self, x_batch, random_t):
        img = x_batch[0]
        # captions = x_batch[1]

        noise = tf.random.normal(tf.shape(img))
        assert(noise.shape == img.shape)

        # shape becomes (batch, 1, 1, 1) for broadcasting purposes
        sqrt_a       = tf.gather(self.sqrt_alphas_c, random_t)[:, None, None, None]
        sqrt_one_a   = tf.gather(self.sqrt_one_minus_alphas_c,random_t)[:, None, None, None] 
        x = sqrt_a * img + sqrt_one_a * noise

        return x, noise




@tf.function
def train_step(x_batch, model, diffUtils, batchsize):
    
    # random time (0 - 1000)
    random_t = tf.random.uniform(shape=[batchsize], minval=0, maxval=1000, dtype=tf.int32)
    
    # create and apply noise linearly (no noise schedueler yet)
    x_batch, noise = diffUtils.apply_noise(x_batch, random_t)

    with tf.GradientTape() as tape:
        pred_noise = model([x_batch, random_t], training=True)
        loss = model.loss(noise, pred_noise)

    gradient = tape.gradient(loss, model.trainable_weights)
    model.optimizer.apply_gradients(zip(gradient, model.trainable_weights))

    return loss


model = U_Net()
du = DiffusionUtils()

# Sampling

In [ ]:
du = DiffusionUtils()
def sample(model, diffUtils):
    x = tf.random.normal((1,64,64,4))

    for t in tqdm(reversed(range(1000))):
        if t > 0:
            z = tf.random.normal((1, 64, 64, 4))
        else:
            z = tf.zeros((1, 64, 64, 4))
        
        time = tf.constant([t], dtype=tf.int32)
        pred_noise = model([x, time], training=False)

        # Fat equation from the DDPM paper
        # x =  tf.gather(diffUtils.one_over_sqrt_alphas, time) * \
        #     (x  - ((1-diffUtils.alphas[t]) * pred_noise /\
        #     diffUtils.sqrt_one_minus_alphas_c[t]))\
        #     + z * np.sqrt(diffUtils.betas[t])
        
        one_over_sqrt_alpha = tf.reshape(tf.gather(diffUtils.one_over_sqrt_alphas, time), [-1, 1, 1, 1])
        alpha_t = tf.reshape(tf.gather(diffUtils.alphas, time), [-1, 1, 1, 1])
        sqrt_one_minus_alpha_c = tf.reshape(tf.gather(diffUtils.sqrt_one_minus_alphas_c, time), [-1, 1, 1, 1])
        beta_t = tf.reshape(tf.gather(diffUtils.betas, time), [-1, 1, 1, 1])

        part1 = one_over_sqrt_alpha
        part2 = x - ((1.0 - alpha_t) * pred_noise / sqrt_one_minus_alpha_c)
        part3 = z * tf.sqrt(beta_t)
        
        x = part1 * part2 + part3
        

    return x


def plot_image(x):
    assert(x.shape[0]==1)
    x = tf.reshape(x,(64,64,4))
    x = (x + 1.0) / 2.0
    x = tf.clip_by_value(x, 0.0, 1.0)
    plt.imshow(x)
    plt.show()

# Training

In [ ]:
""" Making a custom training loop cuz
train_step doesn't have the right
args for a simple model.fit()"""
loss = []

for epoch in range(num_epochs):
    i=0
    for batch in dataset:
        l = train_step(batch, model, du, batchsize)
        loss.append(l)
        
        print(f'epoch:{i}, loss:{l}')
        if i == 1:
            break
        i+=1

        if i % 5:
            plot_image(sample(model, du))

plt.plot(loss)
plt.show()